In [1]:
from hashlib import sha256
from io import BytesIO, StringIO
from pathlib import Path
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from zipfile import ZipFile
import json

import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
raw = Path('../input/kincaid')
out = Path('../output')
source = json.loads((raw / 'source.json').read_text())
cache = raw / 'kinsf6.zip'
keys = ['year', 'month', 'day', 'hour']

In [3]:
cached = cache.is_file()
if cached:
    contents = cache.read_bytes()
else:
    with urlopen(source['url'], timeout=90) as response:
        contents = response.read()
if sha256(contents).hexdigest() != source['sha256']:
    raise ValueError('Kincaid archive SHA-256 differs from the recorded distribution')
if not cached:
    with NamedTemporaryFile(dir=raw, prefix='.kinsf6-', delete=False) as tmp:
        tmp.write(contents)
    Path(tmp.name).replace(cache)

In [4]:
observations = {}
for name, digest in source['observations']['files'].items():
    contents_mvk = (raw / name).read_bytes()
    assert sha256(contents_mvk).hexdigest() == digest
    observations[name] = StringIO(contents_mvk.decode('ascii').replace('\x1a', ''))
peaks = pd.read_csv(observations['SF6_KIN.DAT'], sep=r'\s+', skiprows=1, header=None,
                    names=keys + ['observed_ug_m3', 'arc_km', 'peak_azimuth_deg',
                                  'reported_q_g_s', 'normalized_peak', 'quality'])
sounding = pd.read_csv(observations['MET_K1.DAT'], sep=r'\s+', skiprows=1, header=None,
                       usecols=[0, 1, 2, 3, 11], names=keys + ['mixing_height_m'])
sounding['mixing_height_m'] = sounding['mixing_height_m'].where(sounding['mixing_height_m'] > 0)

with ZipFile(BytesIO(contents)) as archive:
    met_lines = archive.read('ISCST3/KS6-ISC3.MET').decode('ascii').splitlines()[1:]
    emission_lines = archive.read('AERMOD/KINHEMIS.SF6').decode('ascii').splitlines()
    stack_lines = archive.read('AERMOD/KSF6-420.80I').decode('ascii').splitlines()

In [5]:
weather = []
for row in met_lines:
    y, month, day, hour = (int(row[i:i+2]) for i in range(0, 8, 2))
    _, speed, temp, stability, _, _ = map(float, row[8:].split())
    weather.append((y, month, day, hour, speed, temp, int(stability)))
weather = pd.DataFrame(weather, columns=keys + ['wind_m_s', 'ambient_k', 'stability'])
emissions = []
for row in (line for line in emission_lines if line.startswith('SO HOUREMIS')):
    _, _, y, month, day, hour, _, q, temp, speed = row.split()
    emissions.append((int(y), int(month), int(day), int(hour), float(q), float(temp), float(speed)))
emissions = pd.DataFrame(emissions, columns=keys + ['q_g_s', 'stack_k', 'exit_m_s'])
_, _, _, height, _, _, diameter = next(row.split() for row in stack_lines if row.lstrip().startswith('SRCPARAM'))

In [6]:
data = peaks.merge(weather, on=keys, how='left', validate='many_to_one')
data = data.merge(emissions, on=keys, how='left', validate='many_to_one')
data = data.merge(sounding, on=keys, how='left', validate='many_to_one')
assert not data[['wind_m_s', 'ambient_k', 'stability', 'q_g_s', 'stack_k', 'exit_m_s']].isna().any().any()
assert np.allclose(data['reported_q_g_s'], data['q_g_s'], atol=0.11)
data['met_day'] = pd.to_datetime(data[['year', 'month', 'day']].assign(year=data['year'] + 1900)).dt.date
data['stack_height_m'] = float(height)
data['stack_diameter_m'] = float(diameter)
data.to_csv(out / 'kincaid.csv', index=False, float_format='%.7g')
display(data.groupby('year', as_index=False).agg(days=('met_day', 'nunique'), arc_hours=('arc_km', 'size')))

,year,days,arc_hours
0,80,14,694
1,81,10,590
